In [ ]:
!pip install ipywidgets

  PARTE 1 :
- Ativar GPU a ser utilzada para instalação e fine tunning do 'neuralmind/bert-base-portuguese-cased'
- Após o 'nvdia-smi' para ver o uso das placas de vídeo, no momento, a [2], era a menos utilizada com |343MiB /  49140MiB|

In [3]:
import os

# Força o Jupyter a enxergar APENAS a GPU 2 (a que está livre no nvidia-smi)
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch

# Verificando se a configuração funcionou
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0) 
    print(f"GPUs disponíveis para o notebook: {num_gpus}")
    print(f"Placa em uso: {gpu_name}")
else:
    print("CUDA não está disponível. Rodando na CPU.")

GPUs disponíveis para o notebook: 1
Placa em uso: NVIDIA RTX A6000


In [30]:
print(f"Placa em uso: {gpu_name}")

Placa em uso: NVIDIA RTX A6000


PARTE 2:
- CARREGAMENTO DO DATASET;
- AJUSTE DOS DADOS;
- CONVERSÃO PARA COMPATIBILIDADE COM O FORMATO DO HUGGING FACE

In [4]:
import pandas as pd
from datasets import Dataset, DatasetDict

# 2. Carregamento dos dados
df_treino = pd.read_csv('train_bert.csv')
df_dev = pd.read_csv('dev_bert.csv')

In [5]:
print("TREINO\n",df_treino['label'].value_counts())
print("\n\nDEV\n",df_dev['label'].value_counts())

TREINO
 label
0    36090
1      562
Name: count, dtype: int64


DEV
 label
0    9022
1     141
Name: count, dtype: int64


In [6]:
# 3. Preparação das colunas
# Renomeamos 'processed_text' para 'text' para facilitar a integração padrão com a biblioteca Transformers
df_treino = df_treino.rename(columns={'processed_text': 'text'})
df_dev = df_dev.rename(columns={'processed_text': 'text'})

# Isolamos apenas as colunas textuais e de classificação para economizar memória
df_treino = df_treino[['text', 'label']]
df_dev = df_dev[['text', 'label']]

In [26]:
df_treino.columns

Index(['text', 'label'], dtype='str')

In [8]:
# 4. Conversão para o formato Hugging Face
dataset_treino = Dataset.from_pandas(df_treino)
dataset_dev = Dataset.from_pandas(df_dev)

In [9]:
dataset_treino[:1]

{'text': ['Florianópolis tem 23 locais fechados desde fevereiro por descumprirem normas sanitárias Mais de mil ações de vistorias referentes ao cumprimento das medidas sanitárias contra a disseminação da Covid-19 já foram realizadas em Florianópolis desde fevereiro. Neste período, foram 1.074 vistorias, 41 advertências, 258 intimações, 18 multas, 23 locais interditados e 712 estabelecimentos operando dentro das normas, segundo aPrefeitura. Entre os estabelecimentos estãobares, pubs, restaurantes, lojas, shoppings, supermercados, hotéis, além de locais públicos como parques e praias. Somente neste final de semana foram realizadas 129 inspeções, uma advertência, uma multa, 18 intimações e uma interdição cautelar executada. Nas fiscalizações são verificados itens comouso correto de máscaras, distanciamento social, disposição de mesas e cadeiras, disponibilidade de álcool em gel, sistema de ventilação, exposição dos alvarás de funcionamento em locais visíveis e outras regras sanitárias. Co

In [10]:
# 5. Agrupamento estruturado
dataset_hf = DatasetDict({
    'train': dataset_treino,
    'validation': dataset_dev
})

# Verificando a estrutura final carregada
print(dataset_hf) #os dados deixam de ser tabelas comuns e tornam-se tensores prontos para receber a tokenização do BERTimbau.

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 36652
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 9163
    })
})


In [11]:
from transformers import AutoTokenizer

# 1. Instanciando o tokenizador oficial do BERTimbau[cite: 1]
modelo = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(modelo)

# 2. Criando a função de tokenização
def tokenizar_batch(batch):
    # Processa os textos aplicando truncamento e preenchimento até o limite da arquitetura
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# 3. Aplicando a função a todo o dataset[cite: 1]
# O parâmetro batched=True é fundamental aqui. Ele usa a infraestrutura em C++ do Apache Arrow
# para tokenizar milhares de textos por segundo de forma assíncrona.
dataset_tokenizado = dataset_hf.map(tokenizar_batch, batched=True)

# 4. Verificando as novas colunas criadas
print("Colunas originais:", dataset_hf['train'].column_names)
print("Colunas após tokenização:", dataset_tokenizado['train'].column_names)

Map: 100%|██████████| 9163/9163 [00:01<00:00, 5022.46 examples/s]

Colunas originais: ['text', 'label']
Colunas após tokenização: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [38]:
dataset_tokenizado['train']

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 36652
})

In [17]:
print(dataset_tokenizado['train'][2])

{'text': 'Empresa pede anulação do processo do complexo na Serra do Rio do Rastro Contemplada pelas belezas da natureza, a região Sul de Santa Catarina vem crescendo como atração turística. Na Serra do Rio Rastro, uma das propostas é a construção de um complexo turístico. Os primeiros procedimentos licitatórios já foram feitos. Tem empresa vencedora para a construção da estrutura, mas um recurso na justiça determina a paralisação do processo.', 'label': 0, 'input_ids': [101, 18725, 7935, 360, 1163, 171, 1673, 171, 5181, 229, 4553, 171, 1007, 171, 15505, 552, 11433, 350, 15770, 1676, 7828, 22281, 180, 3402, 117, 123, 1079, 1422, 125, 1838, 5193, 3539, 15655, 271, 8988, 18734, 119, 503, 4553, 171, 1007, 15505, 552, 117, 230, 366, 8658, 253, 123, 1803, 125, 222, 5181, 14991, 119, 533, 1867, 12275, 4073, 343, 2926, 770, 506, 5159, 119, 2501, 1799, 12258, 221, 123, 1803, 180, 2388, 117, 449, 222, 8190, 229, 6722, 10439, 123, 13628, 341, 171, 1673, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [39]:
import torch
from transformers import AutoModel

# 1. Carregue o modelo (use o exato modelo BERT que você escolheu no tokenizer)
# Exemplo: 'bert-base-uncased' ou o modelo em português 'neuralmind/bert-base-portuguese-cased'
nome_do_mode = "neuralmind/bert-base-portuguese-cased" 
mode = AutoModel.from_pretrained(nome_do_mode)

# 2. Pegue os dados do exemplo 2 do seu dataset
exemplo = dataset_tokenizado['train'][2]

# 3. Converta as listas do dataset em Tensores do PyTorch adicionando a dimensão de lote [Batch]
input_ids = torch.tensor([exemplo['input_ids']])          # Formato: [1, 512]
attention_mask = torch.tensor([exemplo['attention_mask']])  # Formato: [1, 512]

# 4. Passe pelo BERT sem calcular gradientes (modo de inferência)
with torch.no_grad():
    saidas = mode(input_ids=input_ids, attention_mask=attention_mask)

# 5. Pegue a matriz empilhada de embeddings da última camada
# O formato de 'embeddings_empilhados' será: [1, 512, 768] -> [Lote, Tokens, Colunas]
embeddings_empilhados = saidas.last_hidden_state

# 6. Acesse o primeiro token (índice 0, que possui o ID 101)
embedding_do_id_101 = embeddings_empilhados[0, 0, :]

# Visualização dos resultados
print(f"Formato da matriz empilhada do texto: {embeddings_empilhados.shape}")
print(f"Formato do vetor do token 101: {embedding_do_id_101.shape}")
print("\nPrimeiros 15 valores numéricos (das 768 colunas) do ID 101:")
print(embedding_do_id_101[:15])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20756.65it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Formato da matriz empilhada do texto: torch.Size([1, 512, 768])
Formato do vetor do token 101: torch.Size([768])

Primeiros 15 valores numéricos (das 768 colunas) do ID 101:
tensor([-0.3160, -0.2601,  0.2022,  0.0062,  0.0952, -0.1927,  0.4178,  0.1403,
        -0.0508,  0.2137, -0.2839,  0.4860, -0.6743,  0.1839, -0.0230])


In [45]:
print(embedding_do_id_101[:])

tensor([-3.1600e-01, -2.6006e-01,  2.0223e-01,  6.2470e-03,  9.5165e-02,
        -1.9274e-01,  4.1782e-01,  1.4034e-01, -5.0849e-02,  2.1365e-01,
        -2.8390e-01,  4.8602e-01, -6.7432e-01,  1.8394e-01, -2.2994e-02,
         5.5078e-01, -1.2976e-01,  3.2676e-01,  1.4091e-01, -1.3401e-01,
         3.0808e-02,  1.0131e-01,  5.2959e-01, -2.6197e-01,  3.6128e-01,
         3.8417e-01, -6.0809e-02,  4.8075e-01,  3.4055e-03,  2.4242e-01,
        -2.9915e-01,  5.1963e-01, -1.5015e-03, -3.4309e-01,  8.0024e-02,
        -4.3849e-01,  1.8527e-01, -6.8873e-02, -1.5125e-01,  1.6560e-01,
         2.6615e-01, -1.2152e-01, -2.3464e-01,  3.2199e-01,  1.5715e-01,
        -1.6248e-01, -3.8013e-01, -9.1979e-02,  1.3381e-01,  4.5712e-01,
        -2.2460e-01,  3.5939e-01, -1.4235e-02,  2.3016e-01,  5.2128e-01,
         2.6022e-02,  1.4971e-01, -1.0771e-01, -4.5663e-01, -4.3553e-01,
         3.3616e-01, -1.7220e-01,  4.3269e-01,  5.5971e-02, -4.6291e-01,
        -3.3427e-01,  9.8270e-02, -1.1853e-01,  8.9

In [18]:
#definindo f1-score como metrica princial para o aprendizado

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Definindo a função de avaliação
def compute_metrics(eval_pred):
    # O modelo entrega tuplas contendo suas "certezas" matemáticas (logits) e o gabarito (labels)
    logits, labels = eval_pred
    
    # Extraímos a decisão final do modelo pegando o maior valor (argmax)
    predictions = np.argmax(logits, axis=-1)
    
    # Calculamos as métricas usando a classe minoritária/alvo (fraude = 1) como base
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    
    # O Hugging Face espera receber um dicionário com os resultados
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

1. O que são os logits e o argmax?A rede neural não cospe imediatamente um 0 ou 1. A última camada do modelo (o "cabeçalho" de classificação) produz valores brutos contínuos chamados logits.Para uma notícia específica, o modelo pode gerar um vetor como [4.5, -1.2].O primeiro valor representa a energia/probabilidade de ser 0 (Normal).O segundo valor representa a energia para ser 1 (Fraude).A função np.argmax(logits, axis=-1) percorre esse vetor e escolhe o índice do maior número. Como $4.5 > -1.2$, o argmax retorna o índice 0. É assim que a "opinião" matemática da rede se transforma em uma decisão binária concreta.2. A Matemática das Métricas (Scikit-Learn)Avaliar apenas a Acurácia (a porcentagem de acertos totais) é perigoso em contextos onde fraudes são raras (classes desbalanceadas), pois o modelo poderia "trapacear" chutando 0 para tudo e ainda assim obter 99% de acerto. É por isso que extraímos a Precision e o Recall.Matematicamente, o Scikit-Learn constrói uma Matriz de Confusão para contar os Verdadeiros Positivos ($TP$), Falsos Positivos ($FP$) e Falsos Negativos ($FN$), aplicando as seguintes equações:Precisão (Precision): De todos os contratos que o modelo apontou como fraudulentos, quantos realmente eram? É uma medida de cautela; alta precisão significa poucos alarmes falsos.$$Precision = \frac{TP}{TP + FP}$$Revocação (Recall): De todas as fraudes reais que existiam ocultas nos dados, quantas o modelo conseguiu capturar? É uma medida de sensibilidade; alto recall significa que poucas infrações escaparam do radar.$$Recall = \frac{TP}{TP + FN}$$F1-Score: Como Precision e Recall costumam ser uma balança (quando você aumenta um, o outro tende a cair), o F1-Score calcula a média harmônica entre os dois, criando uma única métrica definitiva de qualidade para o seu modelo.$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

In [ ]:
pip install transformers[torch]

In [20]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Dicionários de mapeamento
id2label = {0: "NORMAL", 1: "FRAUDE"}
label2id = {"NORMAL": 0, "FRAUDE": 1}

# 2. Carregando a arquitetura do modelo
modelo_classificacao = AutoModelForSequenceClassification.from_pretrained(
    modelo, 
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# 3. Definindo as regras do treinamento (COM LOGS ATIVADOS)
training_args = TrainingArguments(
    output_dir='./resultados_bertimbau',
    eval_strategy="epoch",            # Faz uma prova no dev_set ao final de cada época
    save_strategy="epoch",            # Salva o modelo no mesmo intervalo
    logging_strategy="steps",         # Instrui o modelo a registrar logs periodicamente
    logging_steps=200,                # Força a gravação no disco a cada 200 lotes processados
    learning_rate=2e-5,               # Taxa de aprendizado padrão e segura para BERT
    per_device_train_batch_size=16,   # Quantos textos a GPU processa por vez
    per_device_eval_batch_size=16,
    num_train_epochs=3,               # Quantas vezes o modelo lerá os 36 mil textos
    weight_decay=0.01,
    load_best_model_at_end=True,      # Ao final, resgata a melhor versão
    metric_for_best_model="f1",       # CRÍTICO: Avalia o sucesso pela média de acertos de fraude, não pela acurácia geral
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37996.38it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. A Troca de "Cabeçalho" (AutoModelForSequenceClassification)O BERTimbau original passou meses lendo a internet apenas para aprender português (tarefa de Language Modeling). Ao usarmos o AutoModelForSequenceClassification, o código entra na arquitetura original, arranca a última camada neural (o "cabeçalho" que adivinhava palavras) e acopla uma nova camada linear crua com exatamente 2 saídas (num_labels=2). O resto da rede já é fluente em português, mas essa última camada nova não sabe nada; ela será treinada do zero para associar os padrões semânticos às suas fraudes. Ao rodar essa célula, é provável que você veja um aviso em vermelho dizendo que alguns pesos não foram inicializados. Isso é perfeitamente normal e esperado; é o aviso de que o novo cabeçalho cru foi acoplado com sucesso.

2. Tamanho do Lote (batch_size = 16)Como você está trabalhando com textos longos (max_length=512), as matrizes de cálculo ficam enormes. Placas de vídeo comuns dariam erro de falta de memória (Out Of Memory - OOM) instantaneamente. Como você está utilizando a RTX A6000 (que tem uma VRAM gigantesca de 48GB), ela consegue absorver 16 (ou até 32) textos inteiros simultaneamente, paralelizando as multiplicações matriciais e reduzindo drasticamente o tempo de cada época.

3. A Taxa de Aprendizado (learning_rate = 2e-5)Isso é o $0.00002$. Na teoria da descida do gradiente (Gradient Descent), a taxa de aprendizado dita o tamanho dos "passos" que a rede dá para corrigir seus erros. Se for muito grande, o modelo destrói o conhecimento prévio de português do BERT. Se for muito pequena, ele não aprende a detectar as fraudes. Para o BERT, o padrão-ouro validado pela academia fica entre 2e-5 e 5e-5.

Você acaba de concluir toda a preparação estrutural. O próximo e último passo no código é, literalmente, acionar a rede neural.

Execute a célula do seu Jupyter que contém a nossa classe `WeightedLossTrainer` (onde injetamos os pesos para o balanceamento de classes). Logo abaixo dela, crie uma nova célula em branco, insira o comando de ignição e execute-o:

```python
# Inicia o treinamento (Fine-Tuning) na RTX A6000
trainer.train()

```

---


In [32]:
import torch
from torch import nn
from transformers import Trainer

# 1. Definindo os pesos (Enviando o tensor diretamente para a RTX A6000)
# Classe 0 (Normal): Peso 1.0
# Classe 1 (Fraude): Peso 64.2 (resultado de 36090 / 562)
pesos_classes = torch.tensor([1.0, 64.2]).to("cuda:0") # No PyTorch, a GPU isolada na variável de ambiente vira a "cuda:0" local

# 2. Criando a Subclasse Customizada - com **kwargs na memoria
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Extrai o gabarito (labels) do lote atual
        labels = inputs.pop("labels")
        
        # Passa os textos pela rede neural
        outputs = model(**inputs)
        logits = outputs.get("logits")
    
        # Aplica a matemática com penalidade
        loss_fct = nn.CrossEntropyLoss(weight=pesos_classes)
        
        # Calcula a diferença entre a predição (logits) e a realidade (labels)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

# 3. Instanciando o nosso novo Trainer blindado contra desbalanceamento
trainer = WeightedLossTrainer(
    model=modelo_classificacao,
    args=training_args,
    train_dataset=dataset_tokenizado['train'],
    eval_dataset=dataset_tokenizado['validation'],   # Assumindo que seu dataset HF tenha a divisão 'dev'
    compute_metrics=compute_metrics
)

---

### A Teoria em Ação: O que vai acontecer agora?

Assim que você rodar esse comando, a barra de progresso do treinamento aparecerá e o BERTimbau começará o seu ciclo de aprendizado (o *Training Loop*). Por baixo dos panos, o PyTorch executará quatro etapas matemáticas de forma ininterrupta:

**1. O Forward Pass (A Leitura)**
A GPU vai puxar lotes de 16 notícias (os `input_ids` e a `attention_mask`) e passá-los pelas 12 camadas de atenção da arquitetura. Na última camada, o modelo emitirá a sua opinião crua (os *logits*) dizendo se acha que aquele lote é composto por textos normais ou fraudes.

**2. O Cálculo do Erro (A Penalidade)**
Neste momento, a nossa classe `WeightedLossTrainer` intercepta os cálculos. O modelo compara sua própria opinião com o gabarito real (a *label*). Se o modelo tiver classificado uma fraude real como "Normal", o erro matemático será multiplicado por **64.2**. O modelo sentirá um "impacto" enorme em sua função de custo.

**3. O Backpropagation (A Correção)**
Para corrigir esse erro massivo, o otimizador calcula as derivadas parciais (*gradientes*) de trás para frente, percorrendo todos os 110 milhões de parâmetros. Ele ajusta microscopicamente as conexões neurais (usando o tamanho de passo definido pelo seu `learning_rate` de 2e-5) para que, na próxima vez que ler um texto com aquele mesmo vocabulário suspeito, ele acerte a classificação.

**4. A Avaliação (A Prova Final)**
Como configuramos a estratégia de avaliação por época (`epoch`), após ler todas as 36 mil notícias, o modelo pausará o treinamento. Ele pegará o seu `dev_set`, fará previsões sem olhar o gabarito e rodará a nossa função `compute_metrics`.

Uma tabela será impressa na sua tela mostrando exatamente a Acurácia, a Precisão, o Recall e o F1-Score que ele obteve. O processo se repetirá por 3 épocas completas, e ao final, o algoritmo salvará automaticamente a versão da rede que alcançou a melhor média matemática na detecção de fraudes.

In [33]:
# Inicia o treinamento (Fine-Tuning) na RTX A6000
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.223801,0.162780,0.993234,0.781690,0.776224,0.787234
2,0.029487,0.112435,0.993452,0.803922,0.745455,0.872340
3,0.053532,0.255265,0.994980,0.828358,0.874016,0.787234


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.67it/s]


TrainOutput(global_step=6873, training_loss=0.15589919288692977, metrics={'train_runtime': 2564.0846, 'train_samples_per_second': 42.883, 'train_steps_per_second': 2.68, 'total_flos': 2.893063920316416e+16, 'train_loss': 0.15589919288692977, 'epoch': 3.0})

In [4]:
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 48.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 9.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.0/803.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 51.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.6.2
    Uninstalling safetensors-0.6.2:
      Successfully uninstalled safetensors-0.6.2
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.1.9
  

In [1]:
import os, glob, json
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import numpy as np, torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, average_precision_score, roc_auc_score)

# 1. Localiza o melhor checkpoint (melhor F1 no dev), registrado pelo próprio Trainer
ckpts = sorted(glob.glob('./resultados_bertimbau/checkpoint-*'), key=lambda p: int(p.split('-')[-1]))
with open(os.path.join(ckpts[-1], 'trainer_state.json')) as f:
    melhor_ckpt = json.load(f)['best_model_checkpoint']
print("Melhor checkpoint:", melhor_ckpt)

# 2. Tokenizer original + modelo fine-tunado
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
modelo_final = AutoModelForSequenceClassification.from_pretrained(melhor_ckpt)

def tokenizar_batch(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=512)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1, 'precision': p, 'recall': r}

# 3. Conjunto de teste
df_teste = pd.read_csv('test_bert.csv').rename(columns={'processed_text': 'text'})[['text', 'label']]
print("TESTE\n", df_teste['label'].value_counts())
dataset_teste = Dataset.from_pandas(df_teste).map(tokenizar_batch, batched=True)

# 4. Trainer só para inferência (a perda ponderada não é necessária aqui)
avaliador = Trainer(
    model=modelo_final,
    args=TrainingArguments(output_dir='./aval_teste', per_device_eval_batch_size=32, report_to='none'),
    compute_metrics=compute_metrics,
)
saida = avaliador.predict(dataset_teste, metric_key_prefix="test")
print(saida.metrics)

# 5. Métricas detalhadas
y_true, logits = saida.label_ids, saida.predictions
y_pred = logits.argmax(axis=-1)
prob_fraude = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

print("\nMatriz de confusão [linhas=real, colunas=predito]:\n", confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'FRAUDE'], digits=4))
print(f"PR-AUC: {average_precision_score(y_true, prob_fraude):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_true, prob_fraude):.4f}")

df_teste['pred'], df_teste['prob_fraude'] = y_pred, prob_fraude
df_teste.to_csv('predicoes_teste.csv', index=False)

Melhor checkpoint: ./resultados_bertimbau/checkpoint-6873


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TESTE
 label
0    11277
1      176
Name: count, dtype: int64


Map:   0%|          | 0/11453 [00:00<?, ? examples/s]

{'test_loss': 0.030307859182357788, 'test_model_preparation_time': 0.0023, 'test_accuracy': 0.9952850781454641, 'test_f1': 0.8383233532934132, 'test_precision': 0.8860759493670886, 'test_recall': 0.7954545454545454, 'test_runtime': 76.74, 'test_samples_per_second': 149.244, 'test_steps_per_second': 4.665}

Matriz de confusão [linhas=real, colunas=predito]:
 [[11259    18]
 [   36   140]]
              precision    recall  f1-score   support

      NORMAL     0.9968    0.9984    0.9976     11277
      FRAUDE     0.8861    0.7955    0.8383       176

    accuracy                         0.9953     11453
   macro avg     0.9414    0.8969    0.9180     11453
weighted avg     0.9951    0.9953    0.9952     11453

PR-AUC: 0.8956
ROC-AUC: 0.9952
